# Experiment 2 — TF-IDF + GPT-2 surprisal + mean dependency length

Experiment 1 picked **GPT-2 `mean_surprisal`** as the best surprisal.
Here we add **mean dependency length (`mean_mdl`)** and check what each feature adds.

Four feature sets, each run on all three master sheets:

| Feature set | Features |
|---|---|
| `tfidf_only` | TF-IDF |
| `tfidf+surprisal` | TF-IDF + mean_surprisal |
| `tfidf+mdl` | TF-IDF + mean_mdl |
| `tfidf+surprisal+mdl` | TF-IDF + mean_surprisal + mean_mdl |

Metrics: accuracy, precision, recall, F1, AUC and the confusion matrix (positive class = **edit**).

Same 20% test articles as the other notebooks. Results go to `output/paragraph/tables/exp2-tfidf-surprisal-mdl.csv`.

## 1. Settings

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/master/ to read, output/<LEVEL>/tables/ to write
master_dir = Path(f"../../data/{LEVEL}/master")  # master sheets + feature-engineering/
tables_dir = Path(f"../../output/{LEVEL}/tables")  # final result tables
results_path = tables_dir / "exp2-tfidf-surprisal-mdl.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_SIZE = 0.2

# Each feature set = the numeric columns added next to TF-IDF
FEATURE_SETS = {
    "tfidf_only": [],
    "tfidf+surprisal": ["mean_surprisal"],
    "tfidf+mdl": ["mean_mdl"],
    "tfidf+surprisal+mdl": ["mean_surprisal", "mean_mdl"],
}
NUMERIC = ["mean_surprisal", "mean_mdl"]  # every numeric column we need to load

## 2. Train / test split (by article)

In [ ]:
# one shared list of test articles for both levels, so paragraph and sentence results are measured on the same
# articles. Chosen once from the paragraph master sheet with the same SEED / TEST_SIZE as before (so the paragraph
# split is unchanged) and saved; every experiment at every level reads it from then on.
test_articles_file = Path("../../data/test-articles.csv")
if not test_articles_file.exists():
    paragraph_articles = pd.read_csv("../../data/paragraph/master/master-absolute.csv", usecols=["article"])["article"].astype(str)
    unique_paragraph_articles = np.array(sorted(paragraph_articles.unique()))
    n_test = int(round(len(unique_paragraph_articles) * TEST_SIZE))
    chosen = sorted(np.random.default_rng(SEED).permutation(unique_paragraph_articles)[:n_test])
    pd.DataFrame({"article": chosen}).to_csv(test_articles_file, index=False)
test_articles = set(pd.read_csv(test_articles_file)["article"].astype(str))
articles = pd.read_csv(master_dir / "master-absolute.csv", usecols=["article"])["article"].astype(str)
unique_articles = np.array(sorted(articles.unique()))
print("test articles:", len(test_articles & set(unique_articles)), "of", len(unique_articles))

# paragraphs where a metric could not be calculated (one word headings like "Steps") - made by feature-engineering.ipynb
# every pair that has one of these paragraphs (as base or as edit) is left out
undefined = pd.read_csv(master_dir / "feature-engineering" / "undefined_paragraphs.csv")
undefined_keys = set(zip(undefined["article"].astype(str), undefined["revision_group_no"], undefined["revision_depth"]))
print("undefined paragraphs:", len(undefined_keys))

def defined_pairs(df, base_depth_column, edit_depth_column):
    #True for every pair where both paragraphs have all metrics
    base_keys = zip(df["article"].astype(str), df["revision_group_no"], df[base_depth_column])
    edit_keys = zip(df["article"].astype(str), df["revision_group_no"], df[edit_depth_column])
    return np.array([base not in undefined_keys and edit not in undefined_keys for base, edit in zip(base_keys, edit_keys)])

## 3. Helpers

- `add_numeric` scales the numeric columns and glues them next to the TF-IDF columns.
- `train_and_test` fits a logistic regression and returns all the metrics.

In [ ]:
def add_numeric(tfidf_train, tfidf_test, values_train, values_test):
    if values_train.shape[1] == 0:  # tfidf_only: nothing to add
        return tfidf_train, tfidf_test
    scaler = StandardScaler()  # mean 0, std 1 (learned on train only)
    values_train = scaler.fit_transform(values_train)
    values_test = scaler.transform(values_test)
    X_train = hstack([tfidf_train, csr_matrix(values_train)]).tocsr()
    X_test = hstack([tfidf_test, csr_matrix(values_test)]).tocsr()
    return X_train, X_test


def train_and_test(X_train, y_train, X_test, y_test):
    model = LogisticRegression(max_iter=2000, random_state=SEED)
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, predicted).ravel()
    return {
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted),
        "recall": recall_score(y_test, predicted),
        "f1": f1_score(y_test, predicted),
        "auc": roc_auc_score(y_test, probability),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


def show(file, feature_set, m):
    print(f"{file:10s} {feature_set:22s} acc={m['accuracy']:.4f}  prec={m['precision']:.4f}  "
          f"rec={m['recall']:.4f}  f1={m['f1']:.4f}  auc={m['auc']:.4f}  "
          f"[tn={m['tn']} fp={m['fp']} fn={m['fn']} tp={m['tp']}]")


results = []

## 4. Absolute file

Each row is one paragraph, label `edit` = 1, `base` = 0.

In [ ]:
df = pd.read_csv(
    master_dir / "master-absolute.csv",
    usecols=["article", "revision_group_no", "pair_older_depth", "pair_newer_depth", "label", "text", *NUMERIC],
    low_memory=False,
)
keep = defined_pairs(df, "pair_older_depth", "pair_newer_depth")
print("left out (pair has an undefined paragraph):", (~keep).sum(), "rows")
df = df[keep].reset_index(drop=True)
text = df["text"].fillna("").astype(str)
y = (df["label"] == "edit").astype(int).to_numpy()
is_test = df["article"].astype(str).isin(test_articles).to_numpy()
is_train = ~is_test
print("train rows:", is_train.sum(), " test rows:", is_test.sum())

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
tfidf_train = vectorizer.fit_transform(text[is_train])
tfidf_test = vectorizer.transform(text[is_test])

for feature_set, columns in FEATURE_SETS.items():
    values = df[columns].fillna(0).to_numpy(dtype=float)
    X_train, X_test = add_numeric(tfidf_train, tfidf_test, values[is_train], values[is_test])
    m = train_and_test(X_train, y[is_train], X_test, y[is_test])
    show("absolute", feature_set, m)
    results.append({"file": "absolute", "feature_set": feature_set, **m})

## 5. Pair files (cumulative and step)

Same as Experiment 1: show each pair as (A, B) in random order, label = **B is the edit**.

- text: `TF-IDF(B) − TF-IDF(A)`
- numeric: `delta_*` from the sheet (edit − base), flipped when we swapped

In [ ]:
for setting in ["cumulative", "step"]:
    delta_cols = ["delta_" + name for name in NUMERIC]
    df = pd.read_csv(
        master_dir / f"master-pairs-{setting}.csv",
        usecols=["article", "revision_group_no", "base_depth", "edit_depth", "base_text", "edit_text", *delta_cols],
        low_memory=False,
    )
    keep = defined_pairs(df, "base_depth", "edit_depth")
    print("left out (pair has an undefined paragraph):", (~keep).sum(), "pairs")
    df = df[keep].reset_index(drop=True)
    base_text = df["base_text"].fillna("").astype(str).to_numpy(dtype=object)
    edit_text = df["edit_text"].fillna("").astype(str).to_numpy(dtype=object)

    # swap = True -> A is the edit (label 0); swap = False -> B is the edit (label 1)
    swap = np.random.default_rng(SEED).integers(0, 2, size=len(df)).astype(bool)
    A = np.where(swap, edit_text, base_text)
    B = np.where(swap, base_text, edit_text)
    y = (~swap).astype(int)
    sign = np.where(swap, -1.0, 1.0)

    is_test = df["article"].astype(str).isin(test_articles).to_numpy()
    is_train = ~is_test
    print(f"\n=== {setting} ===  train pairs: {is_train.sum()}  test pairs: {is_test.sum()}")

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    vectorizer.fit(list(A[is_train]) + list(B[is_train]))
    tfidf_train = vectorizer.transform(B[is_train]) - vectorizer.transform(A[is_train])
    tfidf_test = vectorizer.transform(B[is_test]) - vectorizer.transform(A[is_test])

    for feature_set, columns in FEATURE_SETS.items():
        cols = ["delta_" + name for name in columns]
        values = df[cols].fillna(0).to_numpy(dtype=float) * sign[:, None]
        X_train, X_test = add_numeric(tfidf_train, tfidf_test, values[is_train], values[is_test])
        m = train_and_test(X_train, y[is_train], X_test, y[is_test])
        show(setting, feature_set, m)
        results.append({"file": setting, "feature_set": feature_set, **m})

## 6. Results

In [ ]:
results_df = pd.DataFrame(results)
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print("saved", results_path)

# Full table: every metric for every file and feature set
results_df.round(4)

In [ ]:
# Accuracy side by side
order = list(FEATURE_SETS)
accuracy_table = results_df.pivot(index="feature_set", columns="file", values="accuracy")
accuracy_table.loc[order, ["absolute", "cumulative", "step"]].round(4)

In [ ]:
# AUC side by side
auc_table = results_df.pivot(index="feature_set", columns="file", values="auc")
auc_table.loc[order, ["absolute", "cumulative", "step"]].round(4)